In [34]:
# ===== Colab 初始化設定 =====
from google.colab import drive
drive.mount('/content/drive')

# 切換到你的專案 test 資料夾（ml_train_ranker.ipynb 所在位置）
import os
os.chdir("/content/drive/MyDrive/Machine Learning and FinTech/Final Project/ml-crypto/test")

# 確認目前所在資料夾與相對位置是否正確
import pathlib
print("Current working directory:", os.getcwd())
print("Parent folder contents:", os.listdir(".."))  # 應該能看到 data/
print("Curated folder contents:", os.listdir("../data/curated"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Current working directory: /content/drive/MyDrive/Machine Learning and FinTech/Final Project/ml-crypto/test
Parent folder contents: ['requirements.txt', 'README.md', 'test', 'src', 'scripts', 'storage', 'notebooks', 'data', 'config', 'models']
Curated folder contents: ['universe_top30_annual.parquet', 'rf_weekly.parquet', 'prices_weekly.parquet', 'factors_weekly.parquet', 'fmp_weekly.parquet', 'ml_preds_weekly.parquet', 'ml_preds_weekly_tft.parquet', 'backtest_ml_longshort.parquet', 'ml_preds_weekly_three.parquet']


In [35]:
import numpy as np, sys
print("python:", sys.version)
print("numpy:", np.__version__)  # 這裡應該要是 1.26.4

python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
numpy: 1.26.4


In [36]:
%pip install --no-cache-dir --only-binary=:all: "catboost==1.2.7"

In [37]:
import numpy, catboost
print("numpy:", numpy.__version__)
print("catboost:", catboost.__version__)
from catboost import CatBoostRanker, Pool  # 能 import 就 OK

numpy: 1.26.4
catboost: 1.2.7


In [16]:
%pip install -U fastparquet

In [38]:
# ml_train_three_model_weighted.ipynb - LightGBM Ranker + CatBoost Ranker + XGBoost Ranker
# with ICIR (signed) — fixed

# %% [markdown]
# 匯入與全域設定
# %%
from pathlib import Path
import pandas as pd
import numpy as np
import json, time, warnings, os, random
from dataclasses import dataclass, asdict
from typing import List, Dict
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")

# ---- 隨機種子（含全域 RNG 供微抖動使用）----
_RNG = None
def set_seed(seed: int = 42):
    global _RNG
    import numpy as _np, random as _random
    _np.random.seed(seed); _random.seed(seed)
    _RNG = _np.random.default_rng(seed)

# 路徑（沿用原檔）
ROOT  = Path("..")
CUR   = ROOT / "data" / "curated"
MET   = ROOT / "data" / "metrics"
MODEL = ROOT / "models"
for p in [CUR, MET, MODEL]:
    p.mkdir(parents=True, exist_ok=True)

# ---- 週內秩轉換：加入極小抖動，降低 ties 對 RankIC 的稀釋 ----
def _rank01_with_noise(s: pd.Series, eps: float = 1e-9) -> pd.Series:
    z = s.astype(float).copy()
    mask = z.notna()
    if mask.any():
        # 用全域 RNG，確保可重現
        z.loc[mask] = z.loc[mask] + _RNG.normal(0.0, eps, int(mask.sum()))
    return z.rank(pct=True, method="average")

def _percentile_rank_by_group(df: pd.DataFrame, group_col: str, score_cols: list) -> pd.DataFrame:
    """對每一週把各模型分數換成 [0,1] 百分位秩（ties=average），並加極小抖動降低 ties。"""
    out = df.copy()
    for c in score_cols:
        out[c + "_rank"] = out.groupby(group_col)[c].transform(lambda x: _rank01_with_noise(x))
    return out

# ---- 逐週 RankIC：對低樣本/低變異週跳過，避免噪音汙染 IR ----
def _weekly_rankic(
    df: pd.DataFrame,
    week_col: str,
    asset_col: str,
    y_col: str,
    rank_cols: list,
    min_group_n: int = 12,      # 比 min_assets_per_week 稍嚴，提升穩健性
    min_unique_y: int = 4       # 當週 y 的有效獨特值太少時，不計 IC
) -> pd.DataFrame:
    """計算每週、每個模型相對於目標 y 的 Spearman RankIC。"""
    tmp = df.copy()
    tmp["_y_rank"] = tmp.groupby(week_col)[y_col].rank(pct=True, method="average")
    recs = []
    for wk, g in tmp.groupby(week_col, sort=True):
        g_eff = g.dropna(subset=[y_col])
        if len(g_eff) < min_group_n or g_eff[y_col].nunique(dropna=True) < min_unique_y:
            # 跳過該週：不足以穩健估 IC
            continue
        for c in rank_cols:
            ic = np.nan
            try:
                ic = spearmanr(g[c], g["_y_rank"], nan_policy="omit").correlation
            except Exception:
                pass
            recs.append((wk, c, ic))
    ic_df = pd.DataFrame(recs, columns=[week_col, "model_col", "IC"]).sort_values([week_col, "model_col"])
    return ic_df

# ---- 滾動 IR 權重（保留正負號）+ 群組回退等權 ----
def _rolling_icir_weights(
    ic_df: pd.DataFrame,
    week_col: str,
    model_col: str,
    win: int = 26,
    min_periods: int = 8
) -> pd.DataFrame:
    """
    對每個模型做滾動 IR = mean(IC)/std(IC)。
    與原版不同：不截斷為 0，而是保留正負號；每週以 L1 規範做有號正規化，總和=1。
    若某週全 NaN 或 |IR| 總和≈0，才回退為等權。
    產出欄位：week_col, model_col, 'w'
    """
    ic_df = ic_df.copy()
    # 逐模型滾動均值與標準差
    ic_df["IC_mean"] = ic_df.groupby(model_col)["IC"].transform(lambda s: s.rolling(win, min_periods=min_periods).mean())
    ic_df["IC_std"]  = ic_df.groupby(model_col)["IC"].transform(lambda s: s.rolling(win, min_periods=min_periods).std())
    ic_df["IR"] = ic_df["IC_mean"] / (ic_df["IC_std"].replace(0, np.nan))

    # 週內帶號 L1 正規化
    ic_df["w"] = ic_df.groupby(week_col)["IR"].transform(
        lambda x: x / (x.abs().sum() + 1e-12)
    )

    # 若該週全 NaN 或 |IR| 總和≈0 → 回退等權
    def _fallback_equal_signed(x: pd.Series) -> pd.Series:
        # 任一可用就保留；若全部不可用或總權重 ~ 0，回退等權
        finite_mask = np.isfinite(x.values)
        if not finite_mask.any() or np.isclose(np.abs(np.nan_to_num(x.values)).sum(), 0.0):
            return pd.Series(np.ones(len(x)) / len(x), index=x.index)
        return x

    ic_df["w"] = ic_df.groupby(week_col)["w"].transform(_fallback_equal_signed)
    return ic_df[[week_col, model_col, "w"]]

def _shift_weights_one_period(w_df: pd.DataFrame, week_col: str) -> pd.DataFrame:
    """把每個模型的權重往後位移一週，用 t-1 權重打 t，避免前視。"""
    w_df = w_df.copy()
    w_df["w_shift"] = w_df.groupby("model_col")["w"].shift(1)
    mask_na = w_df["w_shift"].isna()
    w_df.loc[mask_na, "w_shift"] = w_df.loc[mask_na, "w"]  # 第一週回退到當週（或可改均權）
    w_df = w_df.drop(columns=["w"]).rename(columns={"w_shift": "w"})
    return w_df

# ---- 設定 ----
@dataclass
class CFG:
    lookback_weeks: int = 52
    early_stop_weeks: int = 8
    min_assets_per_week: int = 10
    asset_cap: int = 30
    seed: int = 42

    # 三個 Ranker 的主要超參
    lgbm_params: dict = None
    cat_params: dict = None
    xgb_params: dict = None

    # 集成權重（固定比重；可改 ICIR 自適應）
    w_lgbm: float = 0.4
    w_cat:  float = 0.3
    w_xgb:  float = 0.3

    # 輸出檔
    out_pred: Path = CUR / "ml_preds_weekly_three.parquet"
    out_log:  Path = MET / f"ml_train_logs_three{pd.Timestamp.utcnow().strftime('%Y%m%d')}.json"
    out_imp:  Path = MET / f"ml_feature_importance_three{pd.Timestamp.utcnow().strftime('%Y%m%d')}.csv"

cfg = CFG(
    lgbm_params=dict(
        objective="lambdarank",
        metric="ndcg",
        ndcg_eval_at=[5,10,30],
        boosting_type="gbdt",
        n_estimators=3000,
        learning_rate=0.03,
        num_leaves=63,
        max_depth=-1,
        min_data_in_leaf=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=1.0,
        random_state=42,
        verbose=-1,
        force_row_wise=True,
        label_gain=[0,1,3,7,15],
    ),
    cat_params=dict(
        loss_function="YetiRank",
        eval_metric="NDCG:top=10",
        learning_rate=0.03,
        depth=6,
        l2_leaf_reg=5.0,
        iterations=3000,
        random_seed=42,
        od_type="Iter",          # early stopping
        od_wait=max(20, 16),     # ~2*early_stop_weeks
        verbose=False
    ),
    xgb_params=dict(
        objective="rank:ndcg",
        eval_metric="ndcg@10",
        eta=0.05,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        tree_method="hist",
        nthread=0,
        seed=42,
    )
)

set_seed(cfg.seed)
cfg

CFG(lookback_weeks=52, early_stop_weeks=8, min_assets_per_week=10, asset_cap=30, seed=42, lgbm_params={'objective': 'lambdarank', 'metric': 'ndcg', 'ndcg_eval_at': [5, 10, 30], 'boosting_type': 'gbdt', 'n_estimators': 3000, 'learning_rate': 0.03, 'num_leaves': 63, 'max_depth': -1, 'min_data_in_leaf': 20, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_alpha': 0.0, 'reg_lambda': 1.0, 'random_state': 42, 'verbose': -1, 'force_row_wise': True, 'label_gain': [0, 1, 3, 7, 15]}, cat_params={'loss_function': 'YetiRank', 'eval_metric': 'NDCG:top=10', 'learning_rate': 0.03, 'depth': 6, 'l2_leaf_reg': 5.0, 'iterations': 3000, 'random_seed': 42, 'od_type': 'Iter', 'od_wait': 20, 'verbose': False}, xgb_params={'objective': 'rank:ndcg', 'eval_metric': 'ndcg@10', 'eta': 0.05, 'max_depth': 5, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 5, 'tree_method': 'hist', 'nthread': 0, 'seed': 42}, w_lgbm=0.4, w_cat=0.3, w_xgb=0.3, out_pred=PosixPath('../data/curated/ml_preds_weekly_three.par

In [39]:
# 讀取週資料與宇宙
df  = pd.read_parquet(CUR / "factors_weekly.parquet", engine="fastparquet")
uni = pd.read_parquet(CUR / "universe_top30_annual.parquet", engine="fastparquet")

df["date_week"] = pd.to_datetime(df["date_week"]).dt.tz_localize(None)
uni["symbol"] = uni["symbol"].astype(str).str.upper()

# 資產鍵
ASSET_KEYS = ["market","symbol","coingecko_id"]
asset_key = next((k for k in ASSET_KEYS if k in df.columns), None)
if asset_key is None:
    asset_key = "_asset"; df["_asset"] = "asset_0"
if "symbol" in df.columns:
    df["symbol"] = df["symbol"].astype(str).str.upper()

# 目標：下一週的超額報酬
if "rf_weekly" not in df.columns:
    df["rf_weekly"] = 0.0
df = df.sort_values([asset_key,"date_week"]).copy()
df["excess"] = df["ret_simple_weekly"] - df["rf_weekly"]
df["y_fwd1"] = df.groupby(asset_key)["excess"].shift(-1)

# 特徵集合（優先 *_z；若無則退回原值）
base_cols = [
    "log_mcap_year","log_price","max_price_week",
    "r1","r2","r3","r4","r4_1","rmom3",
    "prcvol_mean_week","prcvol_std_week","vol_4w"
]
fac_cols = [c+"_z" for c in base_cols if c+"_z" in df.columns]
if not fac_cols:
    fac_cols = [c for c in base_cols if c in df.columns]
print("features:", fac_cols)

# 以年匹配的 Top30 限定宇宙
df["year"] = df["date_week"].dt.year
df_u = df.merge(
    uni[["year","symbol"]].drop_duplicates(),
    on=["year","symbol"],
    how="inner"
)

# 簡單 winsorize（可選）再 z-score 已做，這裡不重複；缺值處理在每週過濾
print("rows (universe matched):", len(df_u))
df_u.head(3)

features: ['log_mcap_year_z', 'log_price_z', 'max_price_week_z', 'r1_z', 'r2_z', 'r3_z', 'r4_z', 'r4_1_z', 'rmom3_z', 'prcvol_mean_week_z', 'prcvol_std_week_z', 'vol_4w_z']
rows (universe matched): 5650


,date_week,market,symbol,close,ret_simple_weekly,log_mcap_year,log_price,max_price_week,r1,r2,...,r4_z,r4_1_z,rmom3_z,prcvol_mean_week_z,prcvol_std_week_z,vol_4w_z,rf_weekly,excess,y_fwd1,year
0,2018-04-29,ADA/USDT,ADA,0.36254,0.287429,23.649626,0.309351,0.3866,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.287429,-0.044133,2018
1,2018-05-06,ADA/USDT,ADA,0.34654,-0.044133,23.649626,0.297538,0.3885,0.287429,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,-0.044133,-0.200150,2018
2,2018-05-13,ADA/USDT,ADA,0.27718,-0.200150,23.649626,0.244655,0.3474,-0.044133,0.230611,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,-0.200150,-0.078000,2018


In [40]:
# %% [markdown]
# 每週過濾與宇宙裁切（缺值剔除、資產上限）
# %%
def weekly_tradable(g: pd.DataFrame) -> pd.DataFrame:
    g2 = g.dropna(subset=["y_fwd1"] + fac_cols)
    if len(g2) > cfg.asset_cap:
        if "log_mcap_year" in g2.columns:
            g2 = g2.sort_values("log_mcap_year", ascending=False).head(cfg.asset_cap)
        elif "log_price" in g2.columns:
            g2 = g2.sort_values("log_price", ascending=False).head(cfg.asset_cap)
        else:
            g2 = g2.head(cfg.asset_cap)
    return g2

blocks = []
for d, g in df_u.groupby("date_week"):
    gg = weekly_tradable(g)
    if len(gg) >= cfg.min_assets_per_week:
        blocks.append(gg)

df_tr = pd.concat(blocks, ignore_index=True).sort_values(["date_week", asset_key])
print("filtered rows:", len(df_tr),
      "weeks:", df_tr['date_week'].nunique(),
      "assets:", df_tr[asset_key].nunique())

filtered rows: 5420 weeks: 377 assets: 33


In [41]:
# %% [markdown]
# 排序工具函式（group sizes / RankIC / relevance levels）
# %%
def build_group_sizes(frame: pd.DataFrame) -> np.ndarray:
    grp = frame.groupby("date_week", sort=True).size()
    arr = grp.to_numpy(dtype=np.int32)
    assert int(arr.sum()) == int(len(frame)), "Group sizes do not sum to number of rows."
    return arr

def build_group_qid(frame: pd.DataFrame) -> np.ndarray:
    # XGBoost 需要每一列對應一個 qid（同一週同一 qid）
    week_codes = pd.Categorical(frame["date_week"]).codes
    return week_codes.astype(np.uint32)

def rankic(y_hat, y_true):
    return spearmanr(y_hat, y_true, nan_policy="omit").correlation

def to_relevance_levels(df: pd.DataFrame, n_levels: int = 5, y_col: str = "y_fwd1", group_col: str = "date_week") -> np.ndarray:
    rel = np.empty(len(df), dtype=np.int32)
    for w, idx in df.groupby(group_col).indices.items():
        y = df.loc[idx, y_col].values.astype(float)
        if np.all(np.isnan(y)) or len(y) < 2:
            rel[idx] = n_levels // 2
            continue
        uniq = np.unique(y[~np.isnan(y)])
        if len(uniq) < n_levels:
            ranks = pd.Series(y).rank(method="average", na_option="keep")
            pct = (ranks - 1) / (len(ranks.dropna()) - 1) if len(ranks.dropna()) > 1 else pd.Series(np.full_like(ranks, 0.5))
            bins = np.floor(pct * n_levels).clip(0, n_levels - 1)
            rel[idx] = bins.fillna(n_levels // 2).astype(np.int32).values
        else:
            try:
                q = pd.qcut(y, q=n_levels, labels=False, duplicates="drop")
                maxlev = int(np.nanmax(q))
                if maxlev + 1 < n_levels:
                    q = (q.astype(float) * (n_levels - 1) / max(maxlev, 1)).round().astype("Int64")
                rel[idx] = q.fillna(n_levels // 2).astype(np.int32).values
            except Exception:
                ranks = pd.Series(y).rank(method="average", na_option="keep")
                pct = (ranks - 1) / (len(ranks.dropna()) - 1) if len(ranks.dropna()) > 1 else pd.Series(np.full_like(ranks, 0.5))
                bins = np.floor(pct * n_levels).clip(0, n_levels - 1)
                rel[idx] = bins.fillna(n_levels // 2).astype(np.int32).values
    return rel

In [42]:
# %% [markdown]
# 走勢前移訓練與預測：LightGBM Ranker + CatBoost Ranker + XGBoost Ranker
# %%
import lightgbm as lgb
from catboost import CatBoostRanker, Pool
import xgboost as xgb

set_seed(cfg.seed)

all_dates = sorted(df_tr["date_week"].unique())
pred_rows, log_items = [], []

for i, d in enumerate(all_dates):
    hist_dates = all_dates[:i]
    if len(hist_dates) < cfg.lookback_weeks:
        continue

    train_end = hist_dates[-1]
    train = df_tr[df_tr["date_week"] <= train_end].copy()
    test  = df_tr[df_tr["date_week"] == d].copy()
    if len(test) < cfg.min_assets_per_week:
        continue

    tr_weeks = sorted(train["date_week"].unique())
    if len(tr_weeks) <= cfg.early_stop_weeks + 1:
        continue
    valid_start = tr_weeks[-cfg.early_stop_weeks]

    tr = train[train["date_week"] <  valid_start].copy().reset_index(drop=True)
    va = train[train["date_week"] >= valid_start].copy().reset_index(drop=True)

    # --- 建 ranking 標籤與 group ---
    y_tr_rel = to_relevance_levels(tr, n_levels=5, y_col="y_fwd1", group_col="date_week")
    y_va_rel = to_relevance_levels(va, n_levels=5, y_col="y_fwd1", group_col="date_week")
    g_tr = build_group_sizes(tr)
    g_va = build_group_sizes(va)
    qid_tr = build_group_qid(tr)
    qid_va = build_group_qid(va)

    X_tr, X_va, X_te = tr[fac_cols].values, va[fac_cols].values, test[fac_cols].values

    # --- 1) LightGBM Ranker ---
    try:
        dtr = lgb.Dataset(X_tr, label=y_tr_rel, group=g_tr, free_raw_data=True)
        dva = lgb.Dataset(X_va, label=y_va_rel, group=g_va, free_raw_data=True)
        params = dict(cfg.lgbm_params); params["random_state"] = cfg.seed

        booster_lgb = lgb.train(
            params, dtr,
            valid_sets=[dtr, dva],
            valid_names=["train","valid"],
            num_boost_round=params.get("n_estimators", 3000),
            callbacks=[
                lgb.early_stopping(stopping_rounds=max(20, cfg.early_stop_weeks*2), verbose=False),
                lgb.log_evaluation(period=0),
            ],
        )
        y_hat_lgbm = booster_lgb.predict(X_te, num_iteration=booster_lgb.best_iteration)
    except Exception as e:
        print(f"[WARN] LGBM @ {d} 失敗：{e}")
        booster_lgb = None
        y_hat_lgbm = np.full(len(test), np.nan)

    # --- 2) CatBoost Ranker ---
    try:
        # group_id 需為同週同 id，這裡用週序列號
        week_code_tr = pd.Categorical(tr["date_week"]).codes
        week_code_va = pd.Categorical(va["date_week"]).codes
        week_code_te = pd.Categorical(test["date_week"]).codes  # 只有一週，皆同碼

        cat_model = CatBoostRanker(**cfg.cat_params)
        pool_tr = Pool(X_tr, label=y_tr_rel, group_id=week_code_tr)
        pool_va = Pool(X_va, label=y_va_rel, group_id=week_code_va)
        cat_model.fit(pool_tr, eval_set=pool_va, use_best_model=True, verbose=False)
        y_hat_cat = cat_model.predict(X_te)
    except Exception as e:
        print(f"[WARN] CatBoost @ {d} 失敗：{e}")
        cat_model = None
        y_hat_cat = np.full(len(test), np.nan)

    # --- 3) XGBoost Ranker ---
    try:
        dtrain = xgb.DMatrix(X_tr, label=y_tr_rel)
        dvalid = xgb.DMatrix(X_va, label=y_va_rel)
        dtest  = xgb.DMatrix(X_te)
        dtrain.set_group(g_tr.tolist())
        dvalid.set_group(g_va.tolist())

        evals = [(dtrain, "train"), (dvalid, "valid")]
        xgb_booster = xgb.train(
            params=cfg.xgb_params,
            dtrain=dtrain,
            num_boost_round=cfg.lgbm_params.get("n_estimators", 3000),
            evals=evals,
            early_stopping_rounds=max(20, cfg.early_stop_weeks*2),
            verbose_eval=False
        )
        _bi = getattr(xgb_booster, "best_iteration", None)
        if isinstance(_bi, int) and _bi >= 0:
            y_hat_xgb = xgb_booster.predict(dtest, iteration_range=(0, _bi + 1))
        else:
            y_hat_xgb = xgb_booster.predict(dtest)
    except Exception as e:
        print(f"[WARN] XGBoost @ {d} 失敗：{e}")
        xgb_booster = None
        y_hat_xgb = np.full(len(test), np.nan)

    # --- 收集逐週原始預測（此處不做固定權重加權；集成移到迴圈後） ---
    out = test[["date_week", asset_key, "y_fwd1"]].copy()
    out["pred_lgbm"] = y_hat_lgbm
    out["pred_cat"]  = y_hat_cat
    out["pred_xgb"]  = y_hat_xgb
    pred_rows.append(out)

    # 記錄：各模型 train/valid RankIC（以連續 y_fwd1 為真值）
    rec = {"asof_week": pd.Timestamp(d).strftime("%Y-%m-%d")}
    try:
        rec["train_rankic_lgbm"] = float(rankic(booster_lgb.predict(tr[fac_cols].values, num_iteration=getattr(booster_lgb,"best_iteration", None)), tr["y_fwd1"].values)) if booster_lgb else None
        rec["valid_rankic_lgbm"] = float(rankic(booster_lgb.predict(va[fac_cols].values, num_iteration=getattr(booster_lgb,"best_iteration", None)), va["y_fwd1"].values)) if booster_lgb else None
        rec["lgbm_best_iter"] = int(getattr(booster_lgb, "best_iteration", -1)) if booster_lgb else None
    except: pass
    try:
        rec["train_rankic_cat"] = float(rankic(cat_model.predict(X_tr), tr["y_fwd1"].values)) if cat_model else None
        rec["valid_rankic_cat"] = float(rankic(cat_model.predict(X_va), va["y_fwd1"].values)) if cat_model else None
    except: pass
    try:
        rec["train_rankic_xgb"] = float(rankic(xgb_booster.predict(xgb.DMatrix(tr[fac_cols].values), iteration_range=(0, xgb_booster.best_iteration+1)), tr["y_fwd1"].values)) if xgb_booster else None
        rec["valid_rankic_xgb"] = float(rankic(xgb_booster.predict(xgb.DMatrix(va[fac_cols].values), iteration_range=(0, xgb_booster.best_iteration+1)), va["y_fwd1"].values)) if xgb_booster else None
        rec["xgb_best_iter"] = int(getattr(xgb_booster, "best_iteration", -1)) if xgb_booster else None
    except: pass

    log_items.append(rec)

# === 結果彙整：得到逐週三模型分數 ===
pred = pd.concat(pred_rows, ignore_index=True) if pred_rows else pd.DataFrame()
print("pred rows:", len(pred))
if pred.empty:
    raise SystemExit("No predictions produced. Check filters/parameters.")

# === Rolling ICIR 集成（t-1 權重打 t）===
WEEK_COL   = "date_week"
ASSET_COL  = asset_key
TARGET_COL = "y_fwd1"
score_cols = ["pred_lgbm", "pred_cat", "pred_xgb"]

# 1) 週內轉秩
df_ranked = _percentile_rank_by_group(pred.copy(), WEEK_COL, score_cols)
rank_cols = [c + "_rank" for c in score_cols]

# 2) 逐週 RankIC
ic_weekly = _weekly_rankic(df_ranked, WEEK_COL, ASSET_COL, TARGET_COL, rank_cols)

# 3) 26 週滾動 IR → 權重；並用 t-1 權重打 t
w_rolling = _rolling_icir_weights(ic_weekly, WEEK_COL, "model_col", win=26, min_periods=8)
w_rolling = _shift_weights_one_period(w_rolling, WEEK_COL)

# 4) pivot 成每週 × 模型 權重
w_pivot = w_rolling.pivot(index=WEEK_COL, columns="model_col", values="w").reset_index()
w_pivot = w_pivot.rename(columns={
    "pred_lgbm_rank": "w_lgbm",
    "pred_cat_rank":  "w_cat",
    "pred_xgb_rank":  "w_xgb",
})

# 5) 併回逐筆資料，做「加權秩平均」得到最終 y_pred
df_ens = df_ranked.merge(w_pivot, on=WEEK_COL, how="left")

# 權重缺失（冷啟動）→ 等權
for c in ["w_lgbm", "w_cat", "w_xgb"]:
    if c not in df_ens:
        df_ens[c] = np.nan
row_sum_w = df_ens[["w_lgbm","w_cat","w_xgb"]].sum(axis=1)
for c in ["w_lgbm", "w_cat", "w_xgb"]:
    df_ens.loc[row_sum_w <= 0, c] = 1.0/3.0
    df_ens.loc[row_sum_w >  0, c] = df_ens.loc[row_sum_w > 0, c] / row_sum_w[row_sum_w > 0]

df_ens["y_pred"] = (
    df_ens["pred_lgbm_rank"] * df_ens["w_lgbm"] +
    df_ens["pred_cat_rank"]  * df_ens["w_cat"]  +
    df_ens["pred_xgb_rank"]  * df_ens["w_xgb"]
)

# reverse method : 1 - percentile（ex: PR30 -> PR70）
df_ens["y_pred"] = 1.0 - df_ens["y_pred"]

# 最終輸出給回測/儲存
out_pred = df_ens[[WEEK_COL, ASSET_COL, "y_pred"]].rename(columns={WEEK_COL: "date_week"})

pred rows: 4754


In [45]:
# %% [markdown]
# 評估：Overall RankIC 與逐週 RankIC（ensemble 與各模型）
# %%
from scipy.stats import spearmanr

# --- 1) 相容性處理：欄位命名 ---
_rename_map = {}
if "y_pred_lgbm" in pred.columns: _rename_map["y_pred_lgbm"] = "pred_lgbm"
if "y_pred_cat"  in pred.columns: _rename_map["y_pred_cat"]  = "pred_cat"
if "y_pred_xgb"  in pred.columns: _rename_map["y_pred_xgb"]  = "pred_xgb"
if _rename_map:
    pred = pred.rename(columns=_rename_map)

# --- 2) 對齊 ensemble + 各模型（用 reindex 避免 KeyError）---
keys = ["date_week", asset_key]
pred_for_merge = pred.reindex(columns=keys + ["pred_lgbm", "pred_cat", "pred_xgb"])
eval_df = df_ens.merge(pred_for_merge, on=keys, how="left")

# 合併後再補缺欄，避免取欄位時 KeyError
for c in ["pred_lgbm", "pred_cat", "pred_xgb"]:
    if c not in eval_df.columns:
        eval_df[c] = pd.NA

def _safe_spr(a, b):
    try:
        return spearmanr(a, b, nan_policy="omit").correlation
    except Exception:
        return float("nan")

# --- 3) Overall RankIC ---
overall = {
    "ensemble_RankIC": _safe_spr(eval_df.get("y_pred"),    eval_df.get("y_fwd1")),
    "lgbm_RankIC":     _safe_spr(eval_df.get("pred_lgbm"), eval_df.get("y_fwd1")),
    "cat_RankIC":      _safe_spr(eval_df.get("pred_cat"),  eval_df.get("y_fwd1")),
    "xgb_RankIC":      _safe_spr(eval_df.get("pred_xgb"),  eval_df.get("y_fwd1")),
    "obs":             int(len(eval_df))
}
print("Overall:", overall)

# --- 4) 逐週 RankIC（ensemble）---
rows = []
for d, g in eval_df.groupby("date_week"):
    if len(g) >= cfg.min_assets_per_week:
        rows.append({
            "date_week": d,
            "rankIC": _safe_spr(g.get("y_pred"), g.get("y_fwd1"))
        })
ic_by_week = pd.DataFrame(rows).sort_values("date_week")
display(ic_by_week.describe())

Overall: {'ensemble_RankIC': -0.01779957042359149, 'lgbm_RankIC': nan, 'cat_RankIC': nan, 'xgb_RankIC': nan, 'obs': 4754}


,date_week,rankIC
count,325,325.000000
mean,2022-08-14 00:00:00,-0.065598
min,2019-07-07 00:00:00,-0.802198
25%,2021-01-24 00:00:00,-0.303571
50%,2022-08-14 00:00:00,-0.050605
75%,2024-03-03 00:00:00,0.151648
max,2025-09-21 00:00:00,0.682143
std,NaN,0.305891


In [46]:
# %% [markdown]
# 落檔：predictions / logs / feature importance（LGBM 最後一個 booster）
# %%
# --- 1) 預測輸出 ---
# 若上一格沒定義 out_pred，就從 df_ens 重建
if "out_pred" not in locals() or out_pred is None or out_pred.empty:
    # 確保 WEEK_COL / ASSET_COL 存在（避免 kernel 重啟後未定義）
    if "WEEK_COL" not in locals():
        WEEK_COL = "date_week"
    if "ASSET_COL" not in locals():
        ASSET_COL = asset_key
    out_pred = df_ens[[WEEK_COL, ASSET_COL, "y_pred"]].rename(columns={WEEK_COL: "date_week"})

out_pred.to_parquet(cfg.out_pred, index=False)
print("saved preds:", cfg.out_pred)

# --- 2) 記錄（config + overall 指標 + 訓練歷程） ---
# overall 來自上一格；若不存在則做穩健 fallback
try:
    _overall = overall
except NameError:
    from scipy.stats import spearmanr

    # 相容性：pred 可能還是舊欄名
    _rename_map = {}
    if "y_pred_lgbm" in pred.columns: _rename_map["y_pred_lgbm"] = "pred_lgbm"
    if "y_pred_cat"  in pred.columns: _rename_map["y_pred_cat"]  = "pred_cat"
    if "y_pred_xgb"  in pred.columns: _rename_map["y_pred_xgb"]  = "pred_xgb"
    if _rename_map:
        pred = pred.rename(columns=_rename_map)

    # 缺欄補 NaN，避免 KeyError
    for c in ["pred_lgbm", "pred_cat", "pred_xgb"]:
        if c not in pred.columns:
            pred[c] = pd.NA

    # 與 pred（逐週原始分數）對齊
    keys = ["date_week", ASSET_COL]
    eval_df = df_ens.merge(pred[keys + ["pred_lgbm", "pred_cat", "pred_xgb"]], on=keys, how="left")

    def _safe_spr(a, b):
        try:
            return spearmanr(a, b, nan_policy="omit").correlation
        except Exception:
            return float("nan")

    _overall = {
        "ensemble_RankIC": _safe_spr(eval_df["y_pred"],    eval_df["y_fwd1"]),
        "lgbm_RankIC":     _safe_spr(eval_df["pred_lgbm"], eval_df["y_fwd1"]),
        "cat_RankIC":      _safe_spr(eval_df["pred_cat"],  eval_df["y_fwd1"]),
        "xgb_RankIC":      _safe_spr(eval_df["pred_xgb"],  eval_df["y_fwd1"]),
        "obs":             int(len(eval_df))
    }

log_rec = {"config": asdict(cfg), "overall": _overall, "timeline": log_items}
with open(cfg.out_log, "w") as f:
    json.dump(log_rec, f, indent=2, default=str)
print("saved log:", cfg.out_log)

# --- 3) 特徵重要度（LGBM 最後一次訓練的 booster，如有） ---
try:
    if 'booster_lgb' in locals() and booster_lgb is not None:
        imp_gain  = booster_lgb.feature_importance(importance_type="gain")
        imp_split = booster_lgb.feature_importance(importance_type="split")
        imp = pd.DataFrame({"feature": fac_cols, "gain": imp_gain, "split": imp_split}).sort_values("gain", ascending=False)
        imp.to_csv(cfg.out_imp, index=False)
        display(imp.head(20))
        print("saved importance:", cfg.out_imp)
    else:
        print("importance not available: no booster_lgb")
except Exception as e:
    print("importance not available:", e)

saved preds: ../data/curated/ml_preds_weekly_three.parquet
saved log: ../data/metrics/ml_train_logs_three20251112.json


,feature,gain,split
5,r3_z,435.771420,79
11,vol_4w_z,414.475752,65
9,prcvol_mean_week_z,360.992839,64
4,r2_z,348.955089,52
3,r1_z,308.185091,63
2,max_price_week_z,300.769980,45
10,prcvol_std_week_z,275.300279,44
6,r4_z,274.191439,54
1,log_price_z,269.613411,45
8,rmom3_z,265.450251,46


saved importance: ../data/metrics/ml_feature_importance_three20251112.csv
